In [9]:
# Install dependencies
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q torch
!apt-get -qq install -y ffmpeg

from IPython.display import Javascript, display
from google.colab import files
import whisper, torch

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
# -------------------------------
# Step 1: Record Audio in Browser
# -------------------------------
def record_audio_browser(filename="dialogue.wav", record_seconds=8):
    display(Javascript(f"""
    async function recordAudio() {{
        const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        const recorder = new MediaRecorder(stream);
        let data = [];

        recorder.ondataavailable = event => data.push(event.data);
        recorder.start();

        await new Promise(resolve => setTimeout(resolve, {record_seconds*1000}));
        recorder.stop();

        await new Promise(resolve => recorder.onstop = resolve);

        const blob = new Blob(data, {{ type: 'audio/wav' }});
        const url = URL.createObjectURL(blob);

        // Create download link
        const a = document.createElement("a");
        a.href = url;
        a.download = "{filename}";
        a.textContent = "⬇️ Download your recording (click to save)";
        document.body.appendChild(a);
    }}
    recordAudio();
    """))

print("🎙️ Speak now! Recording will start in your browser.")
record_audio_browser(record_seconds=8)


🎙️ Speak now! Recording will start in your browser.


<IPython.core.display.Javascript object>

In [12]:
# -------------------------------
# Step 2: Upload audio back to Colab
# -------------------------------
print("⬆️ Please upload the downloaded audio file back here...")
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]


⬆️ Please upload the downloaded audio file back here...


Saving dialogue.wav to dialogue.wav


In [13]:
# -------------------------------
# Step 3: Transcribe with Whisper
# -------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Whisper model on {device}...")
whisper_model = whisper.load_model("small", device=device)

# Use 'translate' to always get English text, regardless of input language
result = whisper_model.transcribe(audio_file, task="translate")
transcript = result["text"]
print("📝 Transcript (translated to English):", transcript)


Loading Whisper model on cuda...
📝 Transcript (translated to English):  Hi, this is Soumya and I am here with Rishita. We are working on the project called Dialogs.


In [14]:
# -------------------------------
# Step 4: Summarize with LLM
# -------------------------------
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

summary = summarizer(transcript, max_length=60, min_length=15, do_sample=False)
print("📌 Summary:", summary[0]['summary_text'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Your max_length is set to 60, but your input_length is only 29. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=14)


📌 Summary: Soumya and Rishita are working on the project called Dialogs.
